# Feature Engineering — ML Dataset

> **Múltiples ligas**&nbsp;&nbsp;◦&nbsp;&nbsp;**Temporadas:** 2015-16 → 2023-24&nbsp;&nbsp;◦&nbsp;&nbsp;**Fuente:** core_enriched.parquet

## Objetivos

- Calcular las 9 features que alimentarán el pipeline de modelado.
- Garantizar ausencia de leakage temporal (toda feature usa solo información anterior al partido).
- Gestionar el cold start de los primeros partidos por equipo (NaN en rolling → excluir de ml_dataset).
- Exportar `ml_dataset.parquet` con features + target `ftr`, 0 nulos.

## Estructura del Notebook

| # | Sección | Objetivo |
|---|---|---|
| 0 | Entorno y configuración | Librerías, rutas y parámetros |
| 1 | Carga y ordenación | Leer core_enriched, ordenar por fecha |
| 2 | Features de forma — rolling global | `goal_diff_last5_global`, `xg_diff_last5_global`, `xg_conceded_diff_last5_global`, `sot_diff_last5_global` |
| 3 | Features de forma — rolling por localía | `goal_diff_last5_venue` |
| 4 | Feature ELO | `elo_diff_pre` — rating acumulado histórico partido a partido |
| 5 | Features de clasificación | `points_diff_table`, `venue_win_rate_diff` — temporada actual |
| 6 | Feature de mercado | `prob_diff_market` — cuotas Pinnacle normalizadas por overround |
| 7 | Feature de descanso | `rest_days_diff` |
| 8 | Construcción del `ml_dataset` | Construir ml_dataset con `build_features()`, eliminar cold start |
| 9 | Auditoría | Validaciones de integridad sobre ml_dataset |
| 10 | Conclusiones | Resumen de features y decisiones de diseño |
| 11 | Exportación | Guardado de ml_dataset.parquet |


---
##

## 0) Entorno y configuración

Configuración de dependencias, rutas del proyecto y parámetros de referencia.

In [1]:

from pathlib import Path
import pandas as pd
import sys
import json

# Rutas del proyecto
PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PROCESSED_ROOT   = PROJECT_ROOT / "data" / "processed"
ENRICHED_PATH    = PROCESSED_ROOT / "core_enriched.parquet"
ML_DATASET_PATH  = PROCESSED_ROOT / "ml_dataset.parquet"
ML_SCHEMA_PATH   = PROCESSED_ROOT / "ml_dataset_schema.json"

from src.cleaning import DataValidator
from src.features import (
    build_features,
    compute_elo,
    compute_global_rolling,
    compute_venue_rolling,
    compute_table_features,
    compute_market_feature,
    compute_rest_days,
    check_leakage,
    FEATURES,
    FEATURES_ROLLING,
    WINDOW,
    ELO_K,
    ELO_BASE,
)

### 0.2 Parámetros de referencia

In [2]:
print(f"Rolling window : {WINDOW} partidos")
print(f"ELO K-factor   : {ELO_K}")
print(f"ELO base       : {ELO_BASE}")
print(f"\nFeatures con cold start")
print("------------------------")
print(f"  • Rolling global/localía  → primeros {WINDOW} partidos por equipo")
print(f"  • venue_win_rate_diff     → primera jornada de temporada")
print(f"  • rest_days_diff          → primer partido de temporada")
print(f"\nFeatures sin cold start")
print("------------------------")
print(f"  • elo_diff_pre       → arranca en ELO_BASE={ELO_BASE}")
print(f"  • points_diff_table  → arranca en 0")
print(f"  • prob_diff_market   → cuotas siempre disponibles")

Rolling window : 5 partidos
ELO K-factor   : 20
ELO base       : 1500

Features con cold start
------------------------
  • Rolling global/localía  → primeros 5 partidos por equipo
  • venue_win_rate_diff     → primera jornada de temporada
  • rest_days_diff          → primer partido de temporada

Features sin cold start
------------------------
  • elo_diff_pre       → arranca en ELO_BASE=1500
  • points_diff_table  → arranca en 0
  • prob_diff_market   → cuotas siempre disponibles


---
##

## 1) Carga y ordenación cronológica del dataset

Lectura de `core_enriched.parquet` y ordenación cronológica. El orden temporal es la base para el cálculo de las features rolling, garantizando que cada variable utilice únicamente información previa al partido y evitando cualquier posible data leakage.

In [3]:
df = pd.read_parquet(ENRICHED_PATH)
df = df.sort_values("Date").reset_index(drop=True)

print(f"Dataset: {len(df):,} partidos × {len(df.columns)} columnas")
print(f"Rango:   {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Ligas:   {sorted(df['League'].unique())}")
print(f"Seasons: {sorted(df['Season'].unique())}")

assert df["Date"].is_monotonic_increasing, "⚠ El dataset no está ordenado por fecha"
print("\n✓ Ordenación cronológica verificada")

Dataset: 10,660 partidos × 35 columnas
Rango:   2014-08-16 → 2024-05-26
Ligas:   ['bundesliga', 'laliga', 'premier']
Seasons: ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']

✓ Ordenación cronológica verificada


---
##

## 2) Features de forma reciente global

Cuatro features calculadas sobre los últimos `WINDOW` partidos de cada equipo sin distinguir si se jugaron como local o visitante.

| Feature | Descripción |
|---|---|
| `goal_diff_last5_global` | Diferencia media de goles (marcados − concedidos) en los últimos 5 partidos, equipo local − visitante |
| `xg_diff_last5_global` | Diferencia media de xG (generado − concedido) en los últimos 5 partidos, equipo local − visitante |
| `xg_conceded_diff_last5_global` | Diferencia media de xG concedido en los últimos 5 partidos, equipo local − visitante (indicador defensivo) |
| `sot_diff_last5_global` | Diferencia media de tiros a puerta (a favor − en contra) en los últimos 5 partidos, equipo local − visitante |

In [4]:
df_global = compute_global_rolling(df, WINDOW)
n_nan = df_global["goal_diff_last5_global"].isna().sum()
check_leakage(df_global.reset_index(), df)

print("=== Rolling global ==================================================")
print(f"Shape: {len(df_global):,} filas × {len(df_global.columns)} columnas")
print(f"Cold start (NaN): {n_nan:,} partidos — primeros {WINDOW} sin historial suficiente")
print("✓ Cálculo y leakage verificados")
print("=====================================================================")


=== Rolling global ==================================================
Shape: 10,660 filas × 4 columnas
Cold start (NaN): 319 partidos — primeros 5 sin historial suficiente
✓ Cálculo y leakage verificados


---
##

## 3) Features de forma reciente por localía

Feature calculada separando la condición de local/visitante: el historial del equipo local se calcula usando únicamente sus partidos en casa, y el del visitante usando únicamente sus partidos fuera.

| Feature | Descripción |
|---|---|
| `goal_diff_last5_venue` | Diferencia media de goles (marcados − concedidos) entre los últimos 5 partidos del local en casa y los últimos 5 partidos del visitante fuera |

Esta feature captura el efecto específico de jugar en casa o fuera para cada equipo, complementando su forma reciente general.

In [5]:
df_venue = compute_venue_rolling(df, WINDOW)
n_nan = df_venue["goal_diff_last5_venue"].isna().sum()
check_leakage(df_venue.reset_index(), df)

print("=== Rolling por localía ====================================================================")
print(f"Shape: {len(df_venue):,} filas × {len(df_venue.columns)} columnas")
print(f"Cold start (NaN): {n_nan:,} partidos — primeros {WINDOW} partidos en casa/fuera sin historial suficiente")
print("✓ Cálculo y leakage verificados")
print("============================================================================================")

=== Rolling por localía ====================================================================
Shape: 10,660 filas × 1 columnas
Cold start (NaN): 638 partidos — primeros 5 partidos en casa/fuera sin historial suficiente
✓ Cálculo y leakage verificados


---
##

## 4) Feature ELO

Rating acumulado histórico partido a partido. La feature es el valor **pre-partido**, es decir, se registra antes de procesar el resultado y se actualiza después, lo que garantiza que no haya leakage.

| Feature | Descripción |
|---|---|
| `elo_diff_pre` | Rating ELO del local − rating ELO del visitante antes del partido |

**Fórmula de actualización:**
```
expected_home = 1 / (1 + 10^((elo_away - elo_home) / 400))
delta = K × (resultado_real − expected_home)
elo_home += delta  |  elo_away -= delta
```
donde `resultado_real` = 1 (victoria local), 0.5 (empate), 0 (victoria visitante).

In [6]:
df_elo = compute_elo(df, ELO_K, ELO_BASE)
n_nan  = df_elo["elo_diff_pre"].isna().sum()

print("=== ELO ========================================================")
print(f"Shape: {len(df_elo):,} filas × {len(df_elo.columns)} columnas")
print(f"Cold start (NaN): {n_nan:,} — Todos los equipos arrancan en {ELO_BASE} puntos")
print("✓ Cálculo completado")
print("================================================================")

=== ELO ========================================================
Shape: 10,660 filas × 1 columnas
Cold start (NaN): 0 — Todos los equipos arrancan en 1500 puntos
✓ Cálculo completado


---
##

## 5) Features de clasificación

Dos features que capturan el estado de la temporada actual hasta el momento del partido.

| Feature | Descripción |
|---|---|
| `points_diff_table` | Puntos acumulados en la temporada actual (local − visitante) antes del partido |
| `venue_win_rate_diff` | Win rate en casa del local − win rate fuera del visitante, temporada actual |

In [7]:
df_table = compute_table_features(df)
n_nan_pts = df_table["points_diff_table"].isna().sum()
n_nan_vwr = df_table["venue_win_rate_diff"].isna().sum()
check_leakage(df_table.reset_index(), df)

print("=== Features de clasificación ====================================================")
print(f"Shape: {len(df_table):,} filas × {len(df_table.columns)} columnas")
print(f"Cold start points_diff_table   (NaN): {n_nan_pts:,} partidos - Empiezan todos con 0 puntos")
print(f"Cold start venue_win_rate_diff (NaN): {n_nan_vwr:,} partidos — Primera jornada de temporada")
print("✓ Cálculo y leakage verificados")
print("===================================================================================")

=== Features de clasificación ====================================================
Shape: 10,660 filas × 2 columnas
Cold start points_diff_table   (NaN): 0 partidos - Empiezan todos con 0 puntos
Cold start venue_win_rate_diff (NaN): 295 partidos — Primera jornada de temporada
✓ Cálculo y leakage verificados


---
##

## 6) Feature de mercado

Feature calculada sobre las cuotas de cierre Pinnacle. Sin leakage por definición, ya que las cuotas son información pública antes del partido.

| Feature | Descripción |
|---|---|
| `prob_diff_market` | Probabilidad implícita del local − visitante, normalizada por overround |
```
p_h = 1/PSH  |  p_d = 1/PSD  |  p_a = 1/PSA
overround = p_h + p_d + p_a
prob_diff_market = (p_h / overround) − (p_a / overround)
```

In [8]:
df_market = compute_market_feature(df)
n_nan = df_market["prob_diff_market"].isna().sum()

print("=== Feature de mercado ====================================================")
print(f"Shape: {len(df_market):,} filas × {len(df_market.columns)} columnas")
print(f"Cold start (NaN): {n_nan} — Todas las cuotas disponibles desde el primer partido")
print("✓ Cálculo completado")
print("===========================================================================")

=== Feature de mercado ====================================================
Shape: 10,660 filas × 1 columnas
Cold start (NaN): 0 — Todas las cuotas disponibles desde el primer partido
✓ Cálculo completado


### 6.1 Comparación del overround: Pinnacle vs Bet365

Se analiza empíricamente si Pinnacle presenta un menor overround que Bet365 en nuestro conjunto de datos. Este contraste permite validar su idoneidad como fuente de probabilidades implícitas más eficientes.

In [9]:
overround_ps  = (1/df["PSH"] + 1/df["PSD"] + 1/df["PSA"]).mean()
overround_b365 = (1/df["B365H"] + 1/df["B365D"] + 1/df["B365A"]).mean()

print(f"Overround promedio Pinnacle : {overround_ps:.4f}  (~{(overround_ps-1)*100:.1f}%)")
print(f"Overround promedio Bet365   : {overround_b365:.4f}  (~{(overround_b365-1)*100:.1f}%)")

Overround promedio Pinnacle : 1.0253  (~2.5%)
Overround promedio Bet365   : 1.0491  (~4.9%)


---
##

## 7) Feature de descanso

Feature calculada sobre los partidos de la temporada en curso para cada equipo.

| Feature | Descripción |
|---|---|
| `rest_days_diff` | Días desde el último partido del local − días desde el último partido del visitante |


In [10]:
df_rest = compute_rest_days(df)
n_nan = df_rest["rest_days_diff"].isna().sum()

print("=== Feature de descanso ======================================================")
print(f"Shape: {len(df_rest):,} filas × {len(df_rest.columns)} columnas")
print(f"Cold start (NaN): {n_nan:,} partidos — Primer partido de temporada para cada equipo")
print("✓ Cálculo completado")
print("==============================================================================")

=== Feature de descanso ======================================================
Shape: 10,660 filas × 1 columnas
Cold start (NaN): 295 partidos — Primer partido de temporada para cada equipo
✓ Cálculo completado


---
##

## 8) Construcción del `ml_dataset`

Combinación de todos los bloques en un único dataset mediante `build_features()`. Los partidos con `NaN` en cualquiera de las features con cold start son eliminados del `ml_dataset`.

In [11]:
n_before = len(df)
df_ml = build_features(df)
n_after = len(df_ml)

print(f"Partidos totales      : {n_before:,}")
print(f"Eliminados cold start : {n_before - n_after:,} ({(n_before - n_after)/n_before*100:.1f}%)")
print(f"ml_dataset            : {n_after:,} partidos")

Partidos totales      : 10,660
Eliminados cold start : 868 (8.1%)
ml_dataset            : 9,792 partidos


---
##

## 9) Auditoría del dataset

Validaciones de integridad sobre `ml_dataset` antes de exportar.

In [12]:
print("── Auditoría ml_dataset ─────────────────────────────────────────\n")

nulls_dict = DataValidator.get_nulls_dict(df_ml[FEATURES_ROLLING])
dups       = DataValidator.get_duplicates_count(df_ml, key_col="match_id")
inv_ftr    = DataValidator.get_invalid_categories(df_ml, "ftr", {"H", "D", "A"})

no_nulls  = len(nulls_dict) == 0
no_dups   = dups == 0
valid_ftr = len(inv_ftr) == 0

print(f"  {'✓' if no_nulls  else '✗'} {'Nulos en features':<22} │ {nulls_dict if nulls_dict else 'ninguno'}")
print(f"  {'✓' if no_dups   else '✗'} {'Duplicados match_id':<22} │ {dups}")
print(f"  {'✓' if valid_ftr else '✗'} {'Valores ftr':<22} │ {set(df_ml['ftr'].unique())}")

print("\n── Cobertura por liga ───────────────────────────────────────────\n")
for league, grp in df_ml.groupby("League"):
    seasons = sorted(grp["Season"].unique())
    print(f"  {league:<15} │ {len(grp):>5,} partidos │ temporadas {seasons[0]}–{seasons[-1]}")

all_ok = no_nulls and no_dups and valid_ftr
print("\n───────────────────────────────────────────────────────────────")
print(f"\n{'✓ ml_dataset validado — listo para exportar' if all_ok else '✗ Hay problemas que requieren revisión'}")

── Auditoría ml_dataset ─────────────────────────────────────────

  ✓ Nulos en features      │ ninguno
  ✓ Duplicados match_id    │ 0
  ✓ Valores ftr            │ {'A', 'D', 'H'}

── Cobertura por liga ───────────────────────────────────────────

  bundesliga      │ 2,796 partidos │ temporadas 2015–2024
  laliga          │ 3,511 partidos │ temporadas 2015–2024
  premier         │ 3,485 partidos │ temporadas 2015–2024

───────────────────────────────────────────────────────────────

✓ ml_dataset validado — listo para exportar


---
##

## 10) Conclusiones


`ml_dataset` integra las **10 features** del pipeline de modelado junto al target `ftr`.

### 10.1 Features calculadas

| Feature | Bloque | Cold start | Fuente |
|---|---|---|---|
| `elo_diff_pre` | ELO | No — arranca en `ELO_BASE` | Calculado |
| `goal_diff_last5_global` | Rolling global | Sí | `FTHG`, `FTAG` |
| `xg_diff_last5_global` | Rolling global | Sí | `home_xg`, `away_xg` |
| `xg_conceded_diff_last5_global` | Rolling global | Sí | `home_xg`, `away_xg` |
| `sot_diff_last5_global` | Rolling global | Sí | `HST`, `AST` |
| `goal_diff_last5_venue` | Rolling por localía | Sí | `FTHG`, `FTAG` |
| `points_diff_table` | Clasificación | No — arranca en 0 | `FTR` |
| `venue_win_rate_diff` | Clasificación | Sí | `FTR` |
| `rest_days_diff` | Descanso | Sí | `Date` |
| `prob_diff_market` | Mercado | No | `PSH`, `PSD`, `PSA` |

### 10.2 Garantías

- **Sin leakage** — `shift(1)` en features vectorizadas; valor pre-partido en ELO; cuotas Pinnacle son información pública anterior al partido.
- **Sin nulos** — partidos con cold start eliminados del `ml_dataset`.
- **Sin duplicados** — `match_id` único verificado en auditoría.

### 10.3 Decisiones de diseño

- **Pinnacle vs otras casas** — márgenes más bajos (~1.02–1.03 vs ~1.05 de Bet365), por lo que sus probabilidades implícitas se acercan más a la probabilidad real del partido. Bet365 se incluye como contraste de robustez.
- **Cuotas de cierre vs apertura** — las de cierre incorporan días de flujo de apuestas previo al partido y representan el consenso final del mercado. Las de apertura aún no han procesado esa información.
- **K=20 en ELO** — equilibrio estándar en la literatura: K bajo insensibiliza el rating; K alto lo vuelve volátil.
- **WINDOW=5** — mínimo estadísticamente representativo sin sacrificar demasiados partidos por cold start.

---
##

## 11) Exportación

Guardado de `ml_dataset.parquet` con esquema JSON de referencia.

In [13]:
df_ml.to_parquet(ML_DATASET_PATH, index=False)

schema = {
    "num_rows"           : len(df_ml),
    "num_columns"        : len(df_ml.columns),
    "features"           : FEATURES,
    "target"             : "ftr",
    "target_dist"        : df_ml["ftr"].value_counts().to_dict(),
    "leagues"            : sorted(df_ml["League"].unique().tolist()),
    "seasons"            : sorted(df_ml["Season"].unique().tolist()),
    "dtypes"             : {col: str(df_ml[col].dtype) for col in df_ml.columns},
    "cold_start_dropped" : n_before - n_after,
}

with open(ML_SCHEMA_PATH, "w") as f:
    json.dump(schema, f, indent=2, ensure_ascii=False)

print(f"Dataset ML exportado:")
print(f"  {len(df_ml):,} filas × {len(df_ml.columns)} columnas\n")
print(f"  Columnas:")
print(f"    · Features → {len(FEATURES)}")
print(f"    · Target → 1")
print(f"    · Metadatos → 6 (match_id, League, Season, Date, HomeTeam, AwayTeam)\n")

print(f"Archivos guardados:")
print(f"  · Dataset → {ML_DATASET_PATH.relative_to(PROJECT_ROOT)}")
print(f"  · Esquema → {ML_SCHEMA_PATH.relative_to(PROJECT_ROOT)}")

Dataset ML exportado:
  9,792 filas × 17 columnas

  Columnas:
    · Features → 10
    · Target → 1
    · Metadatos → 6 (match_id, League, Season, Date, HomeTeam, AwayTeam)

Archivos guardados:
  · Dataset → data/processed/ml_dataset.parquet
  · Esquema → data/processed/ml_dataset_schema.json


---
##